# CUB final locked evaluation
Fresh-session workflow. It audits inputs, performs training-only smoke checks, locks the protocol automatically, runs both models sequentially, analyzes paired outputs, and builds a resumable archive. Do not inspect official-test outputs before the lock cell.

In [ ]:
import json, os, platform, subprocess, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)
try:
    import torch
    print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'bf16': torch.cuda.is_available() and torch.cuda.is_bf16_supported()})
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(i, p.name, round(p.total_memory / 2**30, 2), 'GiB')
except Exception as exc:
    print('Torch preflight failed:', repr(exc))

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/Ram21275/newpipeline.git'
BRANCH = 'feat/iclr'
REPO_ROOT = Path('/kaggle/working/newpipeline')
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)
PROJECT = REPO_ROOT / 'projects/logit_evidence_routing'
if not (PROJECT / 'pyproject.toml').is_file():
    raise RuntimeError(f'Project checkout is incomplete: {PROJECT}')
WORK = Path('/kaggle/working/cub_final')
for name in ('environment','manifests','smoke','results','analysis','archive'):
    (WORK / name).mkdir(parents=True, exist_ok=True)
print('PROJECT=', PROJECT)
print('COMMIT=', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('WORK=', WORK)

## Install
This does not reinstall Kaggle's PyTorch/CUDA stack. With Internet disabled, attach compatible wheels and the two model snapshots, install the wheels here, and later add `--local-snapshot` to each model command.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements-cub-final-kaggle.txt')], check=True)
subprocess.run([sys.executable, '-m', 'cub_final', 'inventory', '--output', str(WORK / 'environment/inventory.json')], check=True)

## Input integrity and output manifests
The audit rejects an images-only CUB mirror. Cohort creation reads annotations, never model outputs. The released annotation file's documented 606 extra-zero rows (images 2275 and 9364) are normalized by a narrowly scoped rule and counted in the audit report.

In [ ]:
required_cub_files = [
    'images.txt', 'image_class_labels.txt', 'train_test_split.txt',
    'classes.txt', 'bounding_boxes.txt', 'parts/parts.txt',
    'parts/part_locs.txt', 'attributes/attributes.txt',
    'attributes/certainties.txt',
    'attributes/image_attribute_labels.txt',
]
candidate_roots = sorted({p.parent for p in Path('/kaggle/input').rglob('train_test_split.txt')})
valid_roots = [
    root for root in candidate_roots
    if (root / 'images').is_dir()
    and all((root / name).is_file() for name in required_cub_files)
]
for root in candidate_roots:
    missing = [name for name in required_cub_files if not (root / name).is_file()]
    print({'candidate': str(root), 'images': (root / 'images').is_dir(), 'missing': missing})
if len(valid_roots) != 1:
    raise RuntimeError(f'Expected exactly one complete CUB root, found {valid_roots}')
CUB_ROOT = valid_roots[0]
print('CUB_ROOT=', CUB_ROOT)

def run_visible(command):
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(
            f'Command failed with exit code {completed.returncode}: {command}. '
            'The complete underlying traceback is printed immediately above.'
        )
    return completed

run_visible([sys.executable, '-m', 'cub_final', 'audit', '--cub-root', str(CUB_ROOT), '--output', str(WORK / 'manifests/cub_integrity_audit.json')])
run_visible([sys.executable, '-m', 'cub_final', 'manifests', '--cub-root', str(CUB_ROOT), '--output-dir', str(WORK / 'manifests')])
print((WORK / 'manifests/manifest_summary.json').read_text())

## Training-only real-model smoke checks
Models are loaded sequentially. If unquantized loading fails for memory, rerun both compared paths with the same explicitly recorded `--quantization 4bit` setting.

In [ ]:
run_visible([sys.executable, '-m', 'cub_final', 'smoke', '--architecture', 'llava', '--cub-root', str(CUB_ROOT), '--output', str(WORK / 'smoke/llava_smoke.json')])

In [ ]:
run_visible([sys.executable, '-m', 'cub_final', 'smoke', '--architecture', 'qwen3', '--cub-root', str(CUB_ROOT), '--output', str(WORK / 'smoke/qwen3_smoke.json')])

In [ ]:
for architecture in ('llava', 'qwen3'):
    report = json.loads((WORK / f'smoke/{architecture}_smoke.json').read_text())
    m = report['measurement']
    decisions = json.loads((WORK / 'manifests/manifest_summary.json').read_text())['expensive_decision_count']
    print(architecture, 'single instrumented forward seconds=', round(m['runtime_seconds'], 3), 'visual tokens=', m['visual_token_count'])
    print('rough six-arm hours before batching/resume=', round(m['runtime_seconds'] * decisions * 6 / 3600, 1))

## Automatic lock
This cell will not ask for approval. It fails closed unless both smoke reports passed on official-training images only.

In [ ]:
subprocess.run([sys.executable, '-m', 'cub_final', 'lock', '--manifest-dir', str(WORK / 'manifests'), '--smoke-dir', str(WORK / 'smoke'), '--output', str(WORK / 'FINAL_PROTOCOL.yaml')], check=True)
protocol = json.loads((WORK / 'FINAL_PROTOCOL.yaml').read_text())
assert protocol['status'] == 'LOCKED'
print('protocol_hash=', protocol['protocol_hash'])

## Locked shared experiment: LLaVA then Qwen3
Each condition is an atomic shard. Safe to rerun after interruption. Save notebook output between long phases.

In [ ]:
questions = WORK / 'manifests/question_manifest_expensive.jsonl'
run_visible([sys.executable, '-m', 'cub_final', 'run-shared', '--architecture', 'llava', '--cub-root', str(CUB_ROOT), '--protocol', str(WORK / 'FINAL_PROTOCOL.yaml'), '--questions', str(questions), '--output-dir', str(WORK / 'results')])

In [ ]:
run_visible([sys.executable, '-m', 'cub_final', 'run-shared', '--architecture', 'qwen3', '--cub-root', str(CUB_ROOT), '--protocol', str(WORK / 'FINAL_PROTOCOL.yaml'), '--questions', str(questions), '--output-dir', str(WORK / 'results')])

## Consolidate, cluster-bootstrap, and export
DoLa records retain ordinary generation but mark contrastive scores as non-probabilities. The analysis preserves invalid generated answers.

In [ ]:
combined = WORK / 'results/all_shared_per_example.jsonl'
with combined.open('w', encoding='utf-8') as out:
    for architecture in ('llava', 'qwen3'):
        path = WORK / f'results/{architecture}/shared_per_example_shards.jsonl'
        for line in path.read_text().splitlines():
            if line.strip():
                out.write(line + '\n')
subprocess.run([sys.executable, '-m', 'cub_final', 'analyze', '--input', str(combined), '--output-dir', str(WORK / 'analysis'), '--bootstrap-resamples', '10000'], check=True)
print((WORK / 'analysis/final_analysis.json').read_text()[:5000])
subprocess.run([sys.executable, '-m', 'cub_final', 'figures', '--analysis', str(WORK / 'analysis/final_analysis.json'), '--output-dir', str(WORK / 'analysis/figures')], check=True)

In [ ]:
archive = WORK / 'archive/cub_final_results.tar.gz'
subprocess.run([sys.executable, '-m', 'cub_final', 'archive', '--root', str(WORK), '--include', 'environment', '--include', 'manifests', '--include', 'smoke', '--include', 'FINAL_PROTOCOL.yaml', '--include', 'results', '--include', 'analysis', '--output', str(archive)], check=True)
print('SAVE NOTEBOOK OUTPUT NOW:', archive, Path(str(archive) + '.sha256'))

## Remaining final-paper stages
The notebook above completes the locked shared re-encoding/decoding track. The frozen-feature probes and model-internal selective interventions still use the repository's existing `lger` extractors and must be launched against these locked manifests. They are intentionally not represented as completed by this notebook until their architecture-specific Qwen3 hooks and final-test outputs have passed the same audits.